# pO2 Viewers

Two interactive 2x2 figures. The widgets live in [po2_gui.py](../po2_gui.py), every panel is drawn by [po2_plots.py](../po2_plots.py), and every number comes from [po2_transients.py](../po2_transients.py) (see [01_pipeline_walkthrough.ipynb](01_pipeline_walkthrough.ipynb) for what each function outputs).
- **Single-run viewer** (`RunGUI`): one run at a time, straight from the .mat files (`pt.load_run` → `pt.analyze_run`).
- **All-runs viewer** (`AllRunsGUI`, at the bottom): every run after `pt.process_all_runs`, pooled.

## Single-run viewer

| Panel | Shows |
|---|---|
| [0, 0] | Detection signal of the chosen event method with threshold, event peaks, positive lobes and extraction windows; optionally one other signal on a twin y-axis |
| [1, 0] | Raw or normalized (pO2 / point mean) pO2 of every selected point, chosen with the *pO2 signal* buttons (colors match the bars), with event times as vertical lines; **click an event line to exclude it (gray, window unshaded) or include it again (black)** |
| [1, 1] | RMS (top) and CV (bottom) of each pO2 point; **click a point's column to select / deselect it** (selected = colored with black edge) |
| [0, 1] | The same pO2 signal cut around every included (black) event of [1, 0] for each selected point, in the same colors as [1, 0], with the mean event response (thick black) |

**Controls:** run, event method, second signal, pO2 signal, threshold (typed; press Enter) *relative to RMS*, and *Select all / Deselect all points* (buttons in the figure, above the bar plots). The selection resets when the run changes. Filters, min event spacing and extraction window come from `params` below.

## Before you run this

> **New to Python or Jupyter?** Do the one-time setup in the [README](../README.md#full-setup-from-zero) first: install conda, create the `po2` environment, put the data in `data/`.

1. **Choose the kernel.** Top-right of this notebook in VS Code, click *Select Kernel* → *Python Environments* → **po2**. (Jupyter Lab: *Kernel → Change Kernel → Python (po2)*.) The kernel is the Python process that runs the cells. Picking `po2` makes it use the packages from `environment.yml`.
2. **Run cells top to bottom.** Click in a cell and press **Shift + Enter** to run it and move to the next one, or use **Run All**. Cells share variables like the MATLAB workspace, so a cell only works once the cells above it have run.
3. **Changed a parameter?** Re-run that cell and every cell below it. If things get confusing, **Restart** the kernel (it clears all variables, like `clear all`) and run everything again.
4. Figures and tables appear under the cell that made them. Text printed in red/pink is usually a *warning* (it keeps going), not an *error* (it stops, with a "Traceback" ending in the error message).
5. **Interactive figures need a live kernel.** They do not show on GitHub, and they freeze if the kernel restarts. If a viewer stops responding, re-run its cell.

In [ ]:
# "%matplotlib widget" makes figures INTERACTIVE (clickable, zoomable) using the ipympl package.
# It must run before any figure is made. If figures show as text like "Canvas(...)", restart the kernel
# and run this cell first.
%matplotlib widget
%load_ext autoreload
%autoreload 2

import po2_transients as pt   # DATA layer: every number in the viewers comes from here
import po2_gui as gui         # WIDGET layer: the two viewers (they call pt for data and po2_plots for drawing)

print("Reading session folders from:", pt.DATA_DIR)

## Configuration

In [ ]:
# available data {session folder: [run numbers]}; each folder is inside data/ (or PO2_DATA_DIR)
runs = {
    "26-10-05_B6_318": [5, 6, 8, 9, 11, 12, 14],
    "26-10-07_B6_322": [6, 12],
    "26-10-21_B6_322": [2, 3],
}

# starting values; event_method, height_thresh and rel_to_rms can be changed in the GUI
params = pt.EventParams(
    event_method="sleep_state",  # "pupil", "locom" or "sleep_state" (also switchable in the GUI)
    rel_to_rms=False,
    locom_sigma_s=2.0,         # s
    movmedian_len_s=6,         # s
    lpf_cutoff_hz=0.1,         # Hz
    height_thresh=.004,         # x RMS of the detection signal (or absolute if rel_to_rms=False)
    min_distance_s=10,         # s between events
    extract_window_s=(-20, 20),# s around each event
    neg_peaks = True           # True -> also detect negative peaks (needed for sleep-state transitions in both directions)
)

## Viewer
**Behind the scenes.** `gui.RunGUI` builds the widgets and a 2×2 matplotlib figure. When a control changes, its *callback* (a function in [po2_gui.py](../po2_gui.py)) computes new data with `pt` and redraws only the affected panels with `po2_plots`. A new threshold re-runs `pt.analyze_run` on the cached, already loaded run. Only choosing a new run or event method reads a `.mat` file again.

In [ ]:
viewer = gui.RunGUI(runs, params)
viewer.display()

## All-runs viewer

Pooled over every run in `data` (from `pt.process_all_runs(runs, params)` below).

| Panel | Shows | Computed by |
|---|---|---|
| [0, 0] | Histogram of the *Bin by* metric across all runs, each value colored by the bin it falls in (gray: no bin), with dashed lines at the bin bounds | `pt.bin_edges`, `pt.bin_index` |
| [0, 1] | Mean ± SD of the *Metric* per bin, with every individual value as a dot | `pt.bin_labels` → `pt.bar_values` |
| [1, 0] | Mean ± SD per bin of a *Behavior* signal (top) and of *pO2* (bottom) around the event, after interpolating each window onto a 1 s grid. Event metric: each event counts once. Point metric: each event × point row counts once | `pt.event_responses` → `pt.group_average` |
| [1, 1] | *y* vs *x* scatter colored by bin (black: a dot matching rows of several bins), with a least-squares line and r² | `pt.correlation_values` |

**Bin controls** (above the figure):
- **Bins: + / −** adds a bin / removes the bottom bin. Each bin row has a lower and an upper bound (inclusive; press Enter), labeled with its color. Where bins overlap, the lower row wins.
- **Bin by** picks the metric and the table it comes from: *event* (`event_height`, `event_halfwidth`, `event_auc`; one value per event), *po2_signal* (`rms`, `cv`, `baseline`; one per run x point), *po2_event* (`event_peak`, `peak_abs_height`, `event_peak_time`; one per event x point). The list is `pt.BIN_METRICS`, built from the metric registries, so new metrics show up here automatically.
- **Percentiles**: when checked, the bounds are percentiles of the metric (as in `pt.classify_events`); when unchecked, they are metric values. Toggling converts the typed bounds, so the binning stays the same.

`all_viewer.bins()` and `all_viewer.classify()` (= `pt.classify_events` with the GUI's metric, bins and units) carry the bins downstream.

In [ ]:
# process every run with the starting params (same call as the analysis notebooks; takes a little while)
data = pt.process_all_runs(runs, params)

In [ ]:
all_viewer = gui.AllRunsGUI(data, params, metric="event_auc", percentile=True)
all_viewer.display()

In [ ]:
# downstream use of the GUI's bins: one class per row of the binned metric's table
# (data.events for event metrics, data.po2_signals / data.po2_events for pO2 metrics)
classes = all_viewer.classify()
all_viewer.metric_table()[all_viewer.metric_w.value].groupby(classes, observed=False).describe()